# Energy data review

Inspect **one** built energy product at a time: its summary, map, validation report and (for inferred products) the VIIRS nightlight input.

Dev-only: reads the pipeline's standard outputs; the packaged `energy` model stays visualisation-free. Build products first (see `../01-build-network`).

Run the first code cell to list the available products, then set `PRODUCT`.

In [ ]:
import pathlib
import sys

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
import _helpers as h  # dev-only: reads pipeline outputs, not the energy package

products = h.available_products()
print("Built products:")
display(h.list_products())
for i, name in enumerate(products):
    print(f"  [{i}] {name}")

# >>> pick which product to inspect (change the index) <<<
PRODUCT = products[0]
print("\nSelected PRODUCT =", PRODUCT)

## Summary

In [ ]:
nodes, edges = h.load_layers(PRODUCT)
print(f"{PRODUCT}: {len(nodes)} nodes, {len(edges)} edges, CRS EPSG:{edges.crs.to_epsg()}")
edges["source"].value_counts(dropna=False)

## Map

Edges are coloured by source where few (base), otherwise drawn as thin grey roads; nodes are the observed power terminals (substations / generators). Two-island inferred products look sparse at full extent — use the zoom below.

In [ ]:
h.plot_network(PRODUCT);

## Zoom to one island

For the two-island inferred products, clip to a single island. `h.MAURITIUS_BBOX` and `h.RODRIGUES_BBOX` are provided.

In [ ]:
h.plot_network(PRODUCT, clip=h.MAURITIUS_BBOX, title=f"{PRODUCT} \u2014 Mauritius");

## Validation report

In [ ]:
h.load_validation(PRODUCT)

## Nightlight raster (inferred input)

The inferred products retain OSM roads near VIIRS nightlight targets. Preview the composite if present:

In [ ]:
import rioxarray

tif = h.DATA_ROOT / "incoming/energy/nightlights/viirs-mauritius-rodrigues-2024.tif"
if tif.exists():
    da = rioxarray.open_rasterio(tif, masked=True).squeeze()
    da.plot.imshow(robust=True, figsize=(9, 7))
else:
    print("nightlight composite not found:", tif)